In [1]:
import pandas as pd

base = "../data/retail_clean_dataset/"
for name in ["customer_master", "inventory_snapshot", "promotions",
             "sku_inventory_flags", "sku_master", "store_master"]:
    df = pd.read_csv(base + name + ".csv")
    print(name, df.shape)
    print(df.columns.tolist())
    print()

customer_master (10000, 7)
['cust_id', 'age', 'gender', 'city', 'loyalty_segment', 'preferred_channel', 'registration_date']

inventory_snapshot (21228, 6)
['store_id', 'sku_id', 'stock_on_hand', 'reorder_point', 'safety_stock', 'last_restock_date']

promotions (100, 8)
['promo_id', 'promo_name', 'start_date', 'end_date', 'discount_pct', 'promo_type', 'target_type', 'target_value']

sku_inventory_flags (600, 6)
['sku_id', 'flag', 'affected_stores', 'window_start', 'window_end', 'notes']

sku_master (5000, 7)
['sku_id', 'sku_name', 'category', 'subcategory', 'unit_price', 'cost_price', 'brand']

store_master (30, 5)
['store_id', 'store_name', 'city', 'store_type', 'opening_date']



In [2]:
sales_peek = pd.read_csv(base + "sales_transactions.csv", nrows=5)
print(sales_peek.columns.tolist())
sales_peek

['date', 'receipt_id', 'store_id', 'sku_id', 'customer_id', 'quantity', 'unit_price', 'total_value', 'channel', 'discount_pct', 'promo_id']


,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,promo_id
0,2025-04-02,RCPT00000001,ST16,SKU02498,CUST01410,1,2379.11,2379.11,In-Store,0.0,NaN
1,2025-04-02,RCPT00000001,ST16,SKU04596,CUST01410,3,335.55,1006.65,In-Store,0.0,NaN
2,2022-04-24,RCPT00000002,ST15,SKU00078,CUST00134,1,820.53,820.53,Online,0.0,NaN
3,2022-04-24,RCPT00000002,ST15,SKU00554,CUST00134,2,88.32,176.64,Online,0.0,NaN
4,2024-09-22,RCPT00000003,ST20,SKU03727,CUST08826,1,1660.93,1660.93,Online,0.0,NaN


In [3]:
rows = 0
for chunk in pd.read_csv(base + "sales_transactions.csv", usecols=[0], chunksize=1_000_000):
    rows += len(chunk)
print(rows)

9972038


In [4]:
rows = 0
customers = set()
missing_cust = 0
neg_qty = 0
dates = []

for chunk in pd.read_csv(base + "sales_transactions.csv",
                         usecols=["date", "customer_id", "quantity"],
                         chunksize=1_000_000):
    rows += len(chunk)
    customers.update(chunk["customer_id"].dropna().unique())
    missing_cust += chunk["customer_id"].isna().sum()
    neg_qty += (chunk["quantity"] < 0).sum()
    dates.append(chunk["date"].dropna().min())
    dates.append(chunk["date"].dropna().max())

print("rows:", rows)
print("date range:", min(dates), "to", max(dates))
print("unique customers:", len(customers))
print("missing customer_id:", missing_cust)
print("negative quantities:", neg_qty)

rows: 9972038
date range: 2022-01-01 to 2025-12-31
unique customers: 10000
missing customer_id: 0
negative quantities: 0
